# 08 · Multi-class classification

Homework sub-tasks G, H and I: class-index labels, a `k`-neuron output layer, cross-entropy, accuracy, and a confusion matrix. The shapes and dtypes here are **different from the binary case**, and mixing the two up is the most common source of errors.

Dataset: `rice.csv` — four rice varieties, measured grains. The classes are imbalanced (45 % / 30 % / 18 % / 7 %).

---
## Theory in 60 seconds

- Output layer: **`k` neurons** (one per class), no activation. The outputs are **logits** `z`, shape `(batch, k)`.
- **Softmax** turns logits into probabilities: `pₖ = e^{zₖ} / Σⱼ e^{zⱼ}` — all positive, summing to 1.
- **Cross-entropy** for one sample with true class `c`: `−ln p_c` — again "minus the log of the probability given to the right answer".
- `nn.CrossEntropyLoss` = softmax + log + pick the true class + mean, **all inside the loss**. So: the model outputs raw logits, and the label is the **class index**.

| | binary (07) | multi-class (here) |
|---|---|---|
| output shape | `(batch, 1)` | `(batch, k)` |
| label shape | `(batch, 1)` | `(batch,)` |
| label dtype | `float32` | `int64` |
| loss | `BCEWithLogitsLoss` (or Sigmoid + `BCELoss`) | `CrossEntropyLoss` |
| prediction | `z > 0` | `z.argmax(dim=1)` |

- Softmax is monotonic, so `argmax(logits) == argmax(softmax(logits))`: you need softmax only when you want the **probabilities** themselves.

In [ ]:
import sys; sys.path.insert(0, "..")
import copy
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
from dnd_utils import check, load_rice, RICE_CLASSES

d = load_rice()
X_train, y_train = d["X_train"], d["y_train"]        # (900, 9) float32, (900,) int64 in {0, 1, 2, 3}
X_val, y_val = d["X_val"], d["y_val"]
X_test, y_test = d["X_test"], d["y_test"]
print(X_train.shape, y_train.shape, y_train.dtype, RICE_CLASSES)

---
## Exercise 8.1 · Softmax and cross-entropy by hand

Implement with basic tensor operations only (`exp`, `log`, `sum`, `max`, indexing):

- `my_softmax(z)`: `z` has shape `(n, k)`; returns probabilities `(n, k)`. Make it numerically stable by subtracting each row's maximum before the exponential (this does not change the result — why?).
- `my_cross_entropy(z, y)`: `z` logits `(n, k)`, `y` class indices `(n,)`; returns the mean loss.

<details><summary>Hint — picking the true-class probability of every row</summary>

`p[torch.arange(n), y]` takes element `(i, y[i])` for each row `i` → shape `(n,)`. (The same indexing trick as the third one-hot method in notebook 01.)
</details>

In [ ]:
def my_softmax(z):
    raise NotImplementedError

def my_cross_entropy(z, y):
    raise NotImplementedError

In [ ]:
torch.manual_seed(0)
_z = torch.randn(20, 4) * 2
_y = torch.randint(0, 4, (20,))
check("softmax rows sum to 1", lambda: torch.allclose(my_softmax(_z).sum(dim=1), torch.ones(20)))
check("my_softmax == torch.softmax", lambda: torch.allclose(my_softmax(_z), torch.softmax(_z, dim=1), atol=1e-6))
check("stable for huge logits", lambda: torch.allclose(my_softmax(torch.tensor([[1000.0, 999.0]])), torch.tensor([[0.7311, 0.2689]]), atol=1e-3))
check("my_cross_entropy == nn.CrossEntropyLoss", lambda: torch.allclose(my_cross_entropy(_z, _y), nn.CrossEntropyLoss()(_z, _y), atol=1e-5))

**Reason, then verify.** For a 4-class problem, what is the loss of one sample when

- (a) the logits are all equal (an untrained network, roughly),
- (b) the logits are `[3, 0, 0, 0]` and the true class is 0,
- (c) the logits are `[3, 0, 0, 0]` and the true class is 1?

Give each to 2 decimals.

In [ ]:
loss_a = None
loss_b = None
loss_c = None

In [ ]:
_L = lambda z, y: nn.CrossEntropyLoss()(torch.tensor([z]), torch.tensor([y])).item()
check("(a) ln 4 ≈ 1.39", lambda: abs(loss_a - _L([0., 0., 0., 0.], 2)) < 0.006)
check("(b) ≈ 0.14", lambda: abs(loss_b - _L([3., 0., 0., 0.], 0)) < 0.006)
check("(c) ≈ 3.14", lambda: abs(loss_c - _L([3., 0., 0., 0.], 1)) < 0.006)

Keep `ln k` in mind as the **sanity value of an untrained k-class network**: `ln 2 = 0.69`, `ln 3 = 1.10`, `ln 4 = 1.39`, `ln 10 = 2.30`.

---
## Exercise 8.2 · Which calls work? (dtype and shape of the labels)

`logits` has shape `(6, 4)`. Predict for each label tensor whether `nn.CrossEntropyLoss()(logits, labels)` runs (`True`) or raises an error (`False`). Then run the check and **read the error messages** — you want to recognise them instantly in the test.

In [ ]:
torch.manual_seed(1)
logits = torch.randn(6, 4)
idx = torch.tensor([0, 3, 1, 1, 2, 0])
label_variants = {
    "A  int64, shape (6,)":            idx,
    "B  float32, shape (6,)":          idx.float(),
    "C  int64, shape (6, 1)":          idx[:, None],
    "D  int32, shape (6,)":            idx.to(torch.int32),
    "E  float32 one-hot, shape (6, 4)": torch.eye(4)[idx],
}

In [ ]:
works = {"A": None, "B": None, "C": None, "D": None, "E": None}

In [ ]:
for name, lab in label_variants.items():
    try:
        val = nn.CrossEntropyLoss()(logits, lab).item(); ok = True; msg = f"loss = {val:.4f}"
    except Exception as e:
        ok = False; msg = f"{type(e).__name__}: {str(e)[:90]}"
    check(f"{name:34s} {msg}", lambda: works[name[0]] is ok)

---
## Exercise 8.3 · Class distribution and the baseline

Compute `class_counts` (shape `(4,)`) for the training set and `baseline_acc`: the test accuracy of always predicting the training majority class.

In [ ]:
class_counts = None
baseline_acc = None

In [ ]:
check("class_counts", lambda: list(class_counts) == list(np.bincount(y_train, minlength=4)))
check("baseline_acc", lambda: abs(baseline_acc - (y_test == 0).mean()) < 1e-9)

---
## Exercise 8.4 · Dataset, model, training

1. `ClassificationDataset`: inputs `float32`, labels `int64`; batches `(batch, 9)` and `(batch,)`. Three loaders, batch size 32.
2. `RiceMLP`: `9 → 32 → 16 → 4`, ReLU, **no softmax**.
3. Train a fresh model for 60 epochs with `CrossEntropyLoss` and `Adam(lr=0.001)`; record `train_losses`, `val_losses`, `train_accs`, `val_accs` per epoch, and the validation loss of the untrained model in `initial_val_loss`.

In [ ]:
torch.manual_seed(0)
model = None
initial_val_loss = None
train_losses, val_losses, train_accs, val_accs = [], [], [], []

In [ ]:
check("model outputs (batch, 4) logits", lambda: model(torch.randn(5, 9)).shape == (5, 4))
check("no softmax inside the model (outputs are not rows of probabilities)", lambda: not torch.allclose(model(torch.randn(50, 9)).sum(dim=1), torch.ones(50), atol=1e-3))
check("initial loss is close to ln 4 = 1.386", lambda: abs(initial_val_loss - np.log(4)) < 0.15)
check("training loss went down clearly", lambda: train_losses[-1] < 0.6)
check("validation accuracy above the baseline", lambda: val_accs[-1] > baseline_acc + 0.2)

---
## Exercise 8.5 · Test evaluation, and from logits to probabilities

On the full test set compute:

- `test_logits` `(300, 4)`, `test_true` `(300,)`
- `test_ce`, `test_acc`
- `test_pred` `(300,)`: predicted class indices
- `test_probs` `(300, 4)`: class probabilities

In [ ]:
test_logits = test_true = None
test_ce = test_acc = None
test_pred = None
test_probs = None

In [ ]:
check("shapes", lambda: test_logits.shape == (300, 4) and test_true.shape == (300,) and test_pred.shape == (300,) and test_probs.shape == (300, 4))
check("probabilities sum to 1 per row", lambda: torch.allclose(test_probs.sum(dim=1), torch.ones(300), atol=1e-5))
check("argmax of probabilities == argmax of logits", lambda: torch.equal(test_probs.argmax(dim=1), test_pred))
check("test accuracy well above the baseline", lambda: test_acc > baseline_acc + 0.2)

---
## Exercise 8.6 · The confusion matrix

`cm[i, j]` = number of samples whose **true** class is `i` and **predicted** class is `j`.

Write `confusion_matrix(y_true, y_pred, k)` for NumPy integer arrays — **without a Python loop over the samples**.

<details><summary>Hint 1 — the bincount trick</summary>

Each (true, predicted) pair can be encoded as one integer `true * k + pred` between 0 and k²−1. Count them with `np.bincount(..., minlength=k * k)` and reshape to `(k, k)`.
</details>
<details><summary>Hint 2 — the alternative</summary>

`cm = np.zeros((k, k), dtype=int); np.add.at(cm, (y_true, y_pred), 1)` — adds 1 at every (row, column) pair, correctly handling repeated pairs (plain `cm[y_true, y_pred] += 1` does **not**).
</details>

In [ ]:
def confusion_matrix(y_true, y_pred, k):
    raise NotImplementedError

cm = None       # the confusion matrix of the test set

In [ ]:
_t = np.array([0, 0, 1, 2, 2, 2, 3]); _p = np.array([0, 1, 1, 2, 0, 2, 2])
check("small example", lambda: confusion_matrix(_t, _p, 4).tolist() == [[1, 1, 0, 0], [0, 1, 0, 0], [1, 0, 2, 0], [0, 0, 1, 0]])
check("cm sums to the number of test samples", lambda: cm.shape == (4, 4) and cm.sum() == 300)
check("row sums = number of true samples per class", lambda: (cm.sum(axis=1) == np.bincount(y_test, minlength=4)).all())
check("trace / total = accuracy", lambda: abs(np.trace(cm) / cm.sum() - test_acc) < 1e-6)

Plot it with `plt.imshow`. Requirements (as in the homework): it must be visible which axis is *true* and which is *predicted*, the class names at the ticks, and the count written inside every cell. A small loop over the `k × k` cells for the text is fine.

In [ ]:
# your plot

---
## Exercise 8.7 · Per-class precision, recall, F1 — from the matrix

Vectorised, from `cm` alone: `precision`, `recall`, `f1` as arrays of shape `(4,)`, and `macro_f1` (the plain average of the four F1 values). Where a denominator is 0, the metric must be 0 (no `nan`, no warning).

Which **axis** of `cm` do you sum over for precision, and which for recall? Work it out from the definition of `cm[i, j]` before coding.

In [ ]:
def per_class_metrics(cm):
    """returns precision, recall, f1 -- each of shape (k,)"""
    raise NotImplementedError

precision = recall = f1 = None
macro_f1 = None

In [ ]:
_cm = np.array([[8, 2, 0], [1, 5, 0], [3, 1, 0]])       # class 2 is never predicted
check("small example: precision", lambda: np.allclose(per_class_metrics(_cm)[0], [8 / 12, 5 / 8, 0.0]))
check("small example: recall", lambda: np.allclose(per_class_metrics(_cm)[1], [0.8, 5 / 6, 0.0]))
check("small example: f1 (0 for the never-predicted class, no nan)", lambda: np.allclose(per_class_metrics(_cm)[2], [2 * 8 / (2 * 8 + 4 + 2), 2 * 5 / (2 * 5 + 3 + 1), 0.0]))
check("test set: recall = diagonal / row sums", lambda: np.allclose(recall, np.diag(cm) / cm.sum(axis=1)))
check("macro_f1", lambda: abs(macro_f1 - np.mean(f1)) < 1e-9)

Look at the row of the rarest class (carnaroli). Accuracy barely notices it — why? Which single number in your output would you report to show how well the model treats **all** classes?

✍️ **Your answer** (double-click, write it in your own words *before* opening the solution):

...

---
## Exercise 8.8 · Class weights in `CrossEntropyLoss`

`nn.CrossEntropyLoss(weight=w)` with `w` of shape `(k,)` multiplies the loss of each sample by the weight of **its true class**. Its `mean` reduction is a *weighted* mean:

$$L = \frac{\sum_i w_{y_i}\,\ell_i}{\sum_i w_{y_i}}$$

**a)** Reproduce it by hand with `reduction="none"`.

In [ ]:
def weighted_ce(logits, y, w):
    """logits (n, k), y (n,) int64, w (k,) float tensor -> scalar"""
    raise NotImplementedError

In [ ]:
_w = torch.tensor([1.0, 2.0, 0.5, 4.0])
check("equals nn.CrossEntropyLoss(weight=w)", lambda: torch.allclose(weighted_ce(_z, _y, _w), nn.CrossEntropyLoss(weight=_w)(_z, _y), atol=1e-6))
check("all weights 1 -> ordinary cross-entropy", lambda: torch.allclose(weighted_ce(_z, _y, torch.ones(4)), nn.CrossEntropyLoss()(_z, _y), atol=1e-6))

**b)** Compute "inverse-frequency" weights `class_w = n / (k · count_c)` from the training set (so a class that is half as frequent gets double the weight; a perfectly balanced set would give all ones). Train a fresh `RiceMLP` exactly as before but with the weighted loss, then compute its per-class recall on the test set as `recall_w` (shape `(4,)`) and its accuracy `acc_w`.

In [ ]:
class_w = None        # float32 tensor, shape (4,)
recall_w = None
acc_w = None

In [ ]:
check("class_w = n / (k * count)", lambda: torch.allclose(class_w, torch.tensor(900 / (4 * np.bincount(y_train, minlength=4)), dtype=torch.float32)))
check("the rarest class has the largest weight", lambda: class_w.argmax().item() == 3)
check("recall of the rarest class did not get worse", lambda: recall_w[3] >= recall[3] - 0.05)
check("still far above the baseline", lambda: acc_w > baseline_acc + 0.2)

---
## Exercise 8.9 · Bug hunt: softmax inside the model

A very common mistake: adding `nn.Softmax(dim=1)` as the last layer **and** using `nn.CrossEntropyLoss`. No error, no warning.

**a)** The loss now receives a probability vector (entries in `[0, 1]`, summing to 1) and treats it as logits. For 4 classes, what is the lowest loss a single sample can reach? (The best the model can do is output exactly `[1, 0, 0, 0]` for a class-0 sample.)

In [ ]:
floor_4_classes = None

In [ ]:
check("floor ≈ 0.744", lambda: abs(floor_4_classes - 0.7437) < 1e-3)

**b)** Run both trainings below (identical except for the extra softmax) and compare.

In [ ]:
class RiceMLPBuggy(nn.Module):
    def __init__(self, n_inputs, n_classes):
        super().__init__()
        self.layers = nn.Sequential(nn.Linear(n_inputs, 32), nn.ReLU(), nn.Linear(32, 16), nn.ReLU(),
                                    nn.Linear(16, n_classes), nn.Softmax(dim=1))       # <- the bug
    def forward(self, x):
        return self.layers(x)

try:
    for cls in (RiceMLP, RiceMLPBuggy):
        torch.manual_seed(0)
        m = cls(9, 4); o = torch.optim.Adam(m.parameters(), lr=0.001); hist = []
        for epoch in range(60):
            hist.append(run_epoch(m, loader_train, nn.CrossEntropyLoss(), o)[0])
        vl, va = run_epoch(m, loader_val, nn.CrossEntropyLoss())
        print(f"{cls.__name__:13s} training loss after 1 / 10 / 60 epochs: {hist[0]:.3f} / {hist[9]:.3f} / {hist[-1]:.3f}    validation accuracy {va:.3f}")
except NameError as e:
    print("finish exercise 8.4 first:", e)

The buggy model's accuracy is not catastrophic — so how would you *notice* this bug in the test, from the numbers alone? And why does it train more slowly?

✍️ **Your answer** (double-click, write it in your own words *before* opening the solution):

...

---
## Blank-page drill

New empty notebook, 30 minutes. From `d = load_rice()`:

1. dataset (correct label dtype!) and loaders;
2. model `9 → 30 → 20 → 4`;
3. training loop with loss and accuracy on training and validation data, 60 epochs;
4. plot of the curves with legend;
5. test cross-entropy and accuracy on the full test set;
6. 8 example predictions next to the true classes;
7. confusion matrix computed in NumPy and plotted with labelled axes and counts in the cells.

Afterwards, check yourself against the table at the top of this notebook: output shape, label shape, label dtype, loss, prediction rule — could you fill it in for *both* columns from memory?

In [ ]:
# (do this one in a new notebook)